<a href="https://colab.research.google.com/github/kwalkw/dany/blob/main/Control_difuso_robot_movil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Control difuso de un robot móvil (TurtleBot3 Burger)
**Sistemas Inteligentes** · Google Colab · `scikit-fuzzy`

Un controlador difuso reactivo que lleva al robot a una meta evitando obstáculos. Usa el lidar y el error de rumbo como entradas y entrega `v` y `w`, que son exactamente las dos cosas que acepta `/cmd_vel`.

| # | Ítem del profesor | Sección | Cómo se valida |
|---|---|---|---|
| 1 | Hipótesis: sistema controlado, variable de control, variable de decisión | 1 | Manual (texto) |
| 2 | Variables lingüísticas y análisis de qué controlar | 2 | Manual + auto |
| 3 | Funciones de pertenencia pertinentes (catálogo + las del robot) | 3 y 4 | Auto + gráficas contra los datos |
| 4 | Antecedentes (4 entradas) y consecuentes (2 salidas) | 4 | Auto |
| 5 | Reglas difusas (18, hasta ~20) | 5 | Auto |
| 6 | Sistema de control y simulación | 6 | Auto |
| 7 | Defuzzificación | 7 | Auto |
| 8 | Aplicado a datos / escenarios | 8 | Auto |
| ✔ | Autoverificación final | 9 | Auto |

Las celdas marcadas con ✏️ son las que pueden editar.

In [ ]:
!pip install -q scikit-fuzzy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from skfuzzy import control as ctrl

plt.rcParams["figure.figsize"] = (8, 3.5)
print("scikit-fuzzy", fuzz.__version__)

## 1. Hipótesis ✏️  → *Ítem 1*

- **Sistema controlado:** robot diferencial TurtleBot3 Burger (simulación cinemática; en ROS 2: `/scan`, `/odom` y `/cmd_vel`).
- **Variables de decisión (entradas / antecedentes):** distancia frontal, distancia izquierda, distancia derecha (lidar) y error de rumbo hacia la meta. Son las que se miden y sobre las que se decide.
- **Variables de control (salidas / consecuentes):** velocidad lineal `v` y velocidad angular `w`.
- **Hipótesis:** un controlador difuso de 18 reglas que combina tres distancias del lidar y el error de rumbo permite llevar al robot a una meta evitando obstáculos, sin colisión y respetando `v ≤ 0.22 m/s` y `|w| ≤ 1.0 rad/s`.

⚠️ El profe usó "variable de control" y "variable de decisión" casi como sinónimos. Confirmen la convención con él. El código no depende de eso.

## 2. Análisis: qué variables controlar y cuáles dejar fuera → *Ítem 2*

Un robot diferencial solo recibe **dos** órdenes (`linear.x` y `angular.z`). Todo lo demás es consecuencia de eso o pertenece a otra capa del sistema, así que el alcance se limita a lo que el control difuso puede decidir de forma razonable.

| Variable | Fuente (ROS 2) | Rol | ¿Se incluye? | Razón |
|---|---|---|---|---|
| `d_frente` (mín. lidar ±20°) | `/scan` | entrada | **Sí** | Evita colisión y fija cuánto se puede acelerar |
| `d_izq` (mín. lidar 20°–80°) | `/scan` | entrada | **Sí** | Indica por dónde rodear |
| `d_der` (mín. lidar −80°–−20°) | `/scan` | entrada | **Sí** | Igual que la izquierda |
| `err_rumbo` (ángulo a la meta − yaw) | `/odom` + meta | entrada | **Sí** | Es lo que dirige al robot hacia la meta |
| `v` (m/s) | `/cmd_vel` linear.x | salida | **Sí** | Rango 0 a 0.22 |
| `w` (rad/s) | `/cmd_vel` angular.z | salida | **Sí** | Rango ±1.0 (el hardware llega a 2.84; nos limitamos) |
| Distancia a la meta | `/odom` | no | **No** | Solo como condición de parada (0.15 m); añadirla multiplica las reglas |
| Velocidad medida | `/odom` twist | no | **No** | Añade dinámica sin aportar al objetivo |
| Posición x, y absoluta | `/odom`, `/amcl_pose` | no | **No** | Es consecuencia de v y w; la estimación es de AMCL/EKF |
| Velocidad por rueda / PWM | controlador de motores | no | **No** | Capa de bajo nivel, la resuelve el plugin diff_drive |
| Parámetros de AMCL, Nav2, EKF | otras prácticas | no | **No** | No es control difuso |
| Cámara / seguimiento de carril | `/camera/image_raw` | no | **No** | Ya se cubrió con PID en AutoRace; añade visión |
| Mapa / SLAM / planificación global | Nav2, Cartographer | no | **No** | Un control reactivo no planifica |

**Variables lingüísticas**

| Variable | Rol | Términos |
|---|---|---|
| d_frente | entrada | cerca, media, lejos |
| d_izq, d_der | entrada | cerca, lejos |
| err_rumbo | entrada | derecha, centro, izquierda |
| v | salida | lenta, media, rapida |
| w | salida | giro_der, recto, giro_izq |

**Límite honesto del enfoque:** es un controlador reactivo, así que puede quedarse atascado en obstáculos cóncavos (una U). Para eso existe la planificación global, que queda fuera del alcance.

## 3. Catálogo de funciones de pertenencia → *Ítem 3 (parte 1)*

In [ ]:
x = np.arange(0, 10.01, 0.1)

catalogo = {
    "triangular (trimf)":        fuzz.trimf(x, [0, 5, 10]),
    "trapezoidal (trapmf)":      fuzz.trapmf(x, [2, 4, 6, 8]),
    "sigmoide (sigmf)":          fuzz.sigmf(x, 5, 1.5),
    "función S (smf)":           fuzz.smf(x, 3, 7),
    "función Z (zmf)":           fuzz.zmf(x, 3, 7),
    "función PI (pimf)":         fuzz.pimf(x, 1, 4, 6, 9),
    "gaussiana (gaussmf)":       fuzz.gaussmf(x, 5, 1.2),
    "campana de Bell (gbellmf)": fuzz.gbellmf(x, 2, 4, 5),
}
fig, axs = plt.subplots(2, 4, figsize=(14, 5), sharey=True)
for ax, (nombre, mf) in zip(axs.ravel(), catalogo.items()):
    ax.plot(x, mf, lw=2); ax.set_title(nombre, fontsize=10); ax.set_ylim(-0.05, 1.05)
plt.tight_layout(); plt.show()

a = fuzz.trimf(x, [0, 3, 6]); b = fuzz.trimf(x, [4, 7, 10])
_, y_and = fuzz.fuzzy_and(x, a, x, b)
_, y_or  = fuzz.fuzzy_or(x, a, x, b)
y_not    = fuzz.fuzzy_not(a)
fig, axs = plt.subplots(1, 3, figsize=(14, 3), sharey=True)
for ax, (t, y) in zip(axs, [("A AND B", y_and), ("A OR B", y_or), ("NOT A", y_not)]):
    ax.plot(x, a, "--", alpha=.5); ax.plot(x, b, "--", alpha=.5)
    ax.plot(x, y, lw=2.5); ax.set_title(t)
plt.tight_layout(); plt.show()

catalogo_ok = len(catalogo) >= 8

## 4. Entorno, datos, antecedentes y consecuentes → *Ítems 3 y 4*
Las funciones de pertenencia de este robot **no se sacan de percentiles**: se justifican con valores físicos (radio del robot ≈ 0.105 m, rango del LDS-01 de 0.12 a 3.5 m, velocidad máxima 0.22 m/s). Después se grafican sobre el histograma de los datos para comprobar que cubren donde realmente hay datos.

✏️ Para usar datos reales del rosbag: ejecuten `bag_a_csv.py` en su Ubuntu, suban el CSV y pongan `USAR_CSV = True`.

In [ ]:
# ---------- Parámetros físicos y escenario simulado ----------
V_MAX = 0.22                      # m/s  (Burger; el Waffle llega a 0.26)
W_MAX = 1.0                       # rad/s, límite de seguridad (hardware: 2.84)
R_ROBOT = 0.105                   # m, radio aproximado
RANGO_MIN, RANGO_MAX = 0.12, 3.5  # m, LDS-01
DT = 0.1                          # s
ARENA = (6.0, 6.0)                # m
OBST = [(2.5, 2.0, 0.40), (3.5, 3.5, 0.50), (1.5, 4.0, 0.40),
        (4.4, 2.0, 0.30), (4.3, 4.7, 0.35)]          # (cx, cy, radio)

def envolver(a):
    return (a + np.pi) % (2 * np.pi) - np.pi

def raycast(x, y, ang):
    dx, dy = np.cos(ang), np.sin(ang)
    W, H = ARENA
    with np.errstate(divide="ignore", invalid="ignore"):
        tx = np.where(dx > 1e-9, (W - x) / dx, np.where(dx < -1e-9, -x / dx, np.inf))
        ty = np.where(dy > 1e-9, (H - y) / dy, np.where(dy < -1e-9, -y / dy, np.inf))
    t = np.minimum(tx, ty)
    for cx, cy, r in OBST:
        fx, fy = x - cx, y - cy
        b = fx * dx + fy * dy
        c = fx * fx + fy * fy - r * r
        disc = b * b - c
        t1 = -b - np.sqrt(np.where(disc >= 0, disc, 0.0))
        t = np.where((disc >= 0) & (t1 > 0), np.minimum(t, t1), t)
    return np.clip(t, RANGO_MIN, RANGO_MAX)

SECTORES = {"d_frente": (-20, 20), "d_izq": (20, 80), "d_der": (-80, -20)}   # grados
def lidar(x, y, yaw):
    out = {}
    for nombre, (a0, a1) in SECTORES.items():
        angs = yaw + np.radians(np.linspace(a0, a1, 15))
        out[nombre] = float(raycast(x, y, angs).min())
    return out

# ---------- ✏️ Datos ----------
USAR_CSV = False
RUTA_CSV = "/content/scan_odom.csv"     # generado con bag_a_csv.py
META = (5.0, 5.0)                       # meta (x, y) en el marco de odom

if USAR_CSV:
    raw = pd.read_csv(RUTA_CSV)
    raw["err_rumbo"] = np.degrees(envolver(np.arctan2(META[1] - raw.y, META[0] - raw.x) - raw.yaw))
    df = raw[["d_frente", "d_izq", "d_der", "err_rumbo"]].dropna()
    df[["d_frente", "d_izq", "d_der"]] = df[["d_frente", "d_izq", "d_der"]].clip(RANGO_MIN, RANGO_MAX)
else:
    rng = np.random.default_rng(7)      # datos sintéticos: poses al azar en la arena
    filas = []
    while len(filas) < 1500:
        px, py = rng.uniform(0.3, ARENA[0] - 0.3), rng.uniform(0.3, ARENA[1] - 0.3)
        if any(np.hypot(px - cx, py - cy) < r + 0.3 for cx, cy, r in OBST):
            continue
        yaw = rng.uniform(-np.pi, np.pi)
        s = lidar(px, py, yaw)
        s["err_rumbo"] = np.degrees(envolver(np.arctan2(5.0 - py, 5.0 - px) - yaw))
        filas.append(s)
    df = pd.DataFrame(filas)
print(df.describe().round(2))

# ---------- Antecedentes ----------
d_frente = ctrl.Antecedent(np.linspace(0, RANGO_MAX, 351), "d_frente")
d_izq    = ctrl.Antecedent(np.linspace(0, RANGO_MAX, 351), "d_izq")
d_der    = ctrl.Antecedent(np.linspace(0, RANGO_MAX, 351), "d_der")
err_rumbo = ctrl.Antecedent(np.linspace(-180, 180, 361), "err_rumbo")

u = d_frente.universe
d_frente["cerca"] = fuzz.trapmf(u, [0, 0, 0.30, 0.60])
d_frente["media"] = fuzz.trimf(u,  [0.30, 0.65, 1.20])
d_frente["lejos"] = fuzz.trapmf(u, [0.70, 1.30, RANGO_MAX, RANGO_MAX])
for lat in (d_izq, d_der):
    lat["cerca"] = fuzz.zmf(lat.universe, 0.25, 0.55)
    lat["lejos"] = fuzz.smf(lat.universe, 0.25, 0.55)
ue = err_rumbo.universe
err_rumbo["derecha"]   = fuzz.trapmf(ue, [-180, -180, -45, -5])
err_rumbo["centro"]    = fuzz.gaussmf(ue, 0, 15)
err_rumbo["izquierda"] = fuzz.trapmf(ue, [5, 45, 180, 180])

# ---------- Consecuentes ----------
v_out = ctrl.Consequent(np.linspace(0, V_MAX, 111), "v", defuzzify_method="centroid")
w_out = ctrl.Consequent(np.linspace(-W_MAX, W_MAX, 201), "w", defuzzify_method="centroid")
v_out["lenta"]  = fuzz.trapmf(v_out.universe, [0, 0, 0.03, 0.08])
v_out["media"]  = fuzz.trimf(v_out.universe,  [0.05, 0.11, 0.17])
v_out["rapida"] = fuzz.trapmf(v_out.universe, [0.13, 0.18, V_MAX, V_MAX])
w_out["giro_der"] = fuzz.trapmf(w_out.universe, [-W_MAX, -W_MAX, -0.6, -0.15])
w_out["recto"]    = fuzz.trimf(w_out.universe,  [-0.3, 0, 0.3])
w_out["giro_izq"] = fuzz.trapmf(w_out.universe, [0.15, 0.6, W_MAX, W_MAX])

# ---------- Gráficas: funciones de pertenencia vs. datos ----------
def graficar_variable(var, serie=None):
    fig, ax = plt.subplots()
    if serie is not None:
        ax2 = ax.twinx(); ax2.hist(serie, bins=40, alpha=.2, color="gray")
        ax2.set_ylabel("frecuencia de los datos")
    for nombre, term in var.terms.items():
        ax.plot(var.universe, term.mf, lw=2, label=nombre)
    ax.set_title(f"Funciones de pertenencia: {var.label}")
    ax.set_ylabel("μ"); ax.legend(loc="upper right")
    plt.tight_layout(); plt.show()

for v in [d_frente, d_izq, d_der, err_rumbo]:
    graficar_variable(v, df[v.label])
graficar_variable(v_out); graficar_variable(w_out)

## 5. Reglas difusas ✏️ → *Ítem 5*
18 reglas en cuatro grupos, con prioridad de seguridad: (1) obstáculo al frente, (2) obstáculo lateral, (3) pasillo, (4) ir a la meta. Convención: `err_rumbo > 0` significa que la meta está a la izquierda y `w > 0` es giro a la izquierda.

In [ ]:
F, L, R, TH = d_frente, d_izq, d_der, err_rumbo
V, W = v_out, w_out

reglas = [
    # --- 1. Obstáculo al frente: bajar la velocidad y girar hacia el lado libre ---
    ctrl.Rule(F["cerca"] & R["cerca"] & L["lejos"],                      [V["lenta"], W["giro_izq"]]),  # 1
    ctrl.Rule(F["cerca"] & L["cerca"] & R["lejos"],                      [V["lenta"], W["giro_der"]]),  # 2
    ctrl.Rule(F["cerca"] & L["cerca"] & R["cerca"],                      [V["lenta"], W["giro_izq"]]),  # 3
    ctrl.Rule(F["cerca"] & L["lejos"] & R["lejos"] & TH["izquierda"],    [V["lenta"], W["giro_izq"]]),  # 4
    ctrl.Rule(F["cerca"] & L["lejos"] & R["lejos"] & TH["derecha"],      [V["lenta"], W["giro_der"]]),  # 5
    ctrl.Rule(F["cerca"] & L["lejos"] & R["lejos"] & TH["centro"],       [V["lenta"], W["giro_izq"]]),  # 6
    # --- 2. Obstáculo lateral: alejarse sin frenar del todo ---
    ctrl.Rule(F["media"] & L["cerca"],                                   [V["media"], W["giro_der"]]),  # 7
    ctrl.Rule(F["media"] & R["cerca"],                                   [V["media"], W["giro_izq"]]),  # 8
    ctrl.Rule(F["lejos"] & L["cerca"] & R["lejos"],                      [V["media"], W["giro_der"]]),  # 9
    ctrl.Rule(F["lejos"] & R["cerca"] & L["lejos"],                      [V["media"], W["giro_izq"]]),  # 10
    # --- 3. Pasillo: ambos lados cerca, avanzar recto ---
    ctrl.Rule(F["lejos"] & L["cerca"] & R["cerca"],                      [V["media"], W["recto"]]),     # 11
    ctrl.Rule(F["media"] & L["cerca"] & R["cerca"],                      [V["lenta"], W["recto"]]),     # 12
    # --- 4. Camino libre: ir hacia la meta ---
    ctrl.Rule(F["lejos"] & L["lejos"] & R["lejos"] & TH["centro"],       [V["rapida"], W["recto"]]),    # 13
    ctrl.Rule(F["lejos"] & L["lejos"] & R["lejos"] & TH["izquierda"],    [V["media"], W["giro_izq"]]),  # 14
    ctrl.Rule(F["lejos"] & L["lejos"] & R["lejos"] & TH["derecha"],      [V["media"], W["giro_der"]]),  # 15
    ctrl.Rule(F["media"] & L["lejos"] & R["lejos"] & TH["centro"],       [V["media"], W["recto"]]),     # 16
    ctrl.Rule(F["media"] & L["lejos"] & R["lejos"] & TH["izquierda"],    [V["lenta"], W["giro_izq"]]),  # 17
    ctrl.Rule(F["media"] & L["lejos"] & R["lejos"] & TH["derecha"],      [V["lenta"], W["giro_der"]]),  # 18
]
print(f"Reglas definidas: {len(reglas)}")

## 6. Sistema de control y simulación → *Ítem 6*
El lazo completo: lidar simulado → controlador difuso → `(v, w)` → cinemática diferencial → nueva pose. Es el mismo ciclo que haría un nodo ROS 2 que se suscribe a `/scan` y `/odom` y publica en `/cmd_vel`.

In [ ]:
sistema = ctrl.ControlSystem(reglas)

def ejecutar(inicio, meta, metodo="centroid", max_pasos=1500):
    v_out.defuzzify_method = metodo
    w_out.defuzzify_method = metodo
    sim = ctrl.ControlSystemSimulation(sistema)
    x, y, yaw = inicio
    reg = {k: [] for k in ["x", "y", "v", "w", "d_frente", "d_izq", "d_der", "err"]}
    llego = colision = False
    holgura_min = np.inf
    for _ in range(max_pasos):
        if np.hypot(meta[0] - x, meta[1] - y) < 0.15:
            llego = True; break
        s = lidar(x, y, yaw)
        err = np.degrees(envolver(np.arctan2(meta[1] - y, meta[0] - x) - yaw))
        sim.input["d_frente"] = s["d_frente"]; sim.input["d_izq"] = s["d_izq"]
        sim.input["d_der"] = s["d_der"];       sim.input["err_rumbo"] = err
        sim.compute()
        v, w = sim.output["v"], sim.output["w"]
        for k, val in zip(reg, [x, y, v, w, s["d_frente"], s["d_izq"], s["d_der"], err]):
            reg[k].append(val)
        x += v * np.cos(yaw) * DT; y += v * np.sin(yaw) * DT; yaw = envolver(yaw + w * DT)
        holgura = min(min(np.hypot(x - cx, y - cy) - r for cx, cy, r in OBST),
                      x, ARENA[0] - x, y, ARENA[1] - y) - R_ROBOT
        holgura_min = min(holgura_min, holgura)
        if holgura < 0:
            colision = True; break
    v_out.defuzzify_method = "centroid"; w_out.defuzzify_method = "centroid"
    r = {k: np.array(val) for k, val in reg.items()}
    r.update(llego=llego, colision=colision, holgura_min=holgura_min, tiempo=len(r["x"]) * DT,
             longitud=float(np.sum(np.hypot(np.diff(r["x"]), np.diff(r["y"])))) if len(r["x"]) > 1 else 0.0)
    return r

def dibujar(ax, r, titulo, meta, inicio):
    for cx, cy, rad in OBST:
        ax.add_patch(plt.Circle((cx, cy), rad, color="tab:red", alpha=.35))
    ax.plot(r["x"], r["y"], lw=2); ax.plot(*inicio[:2], "go"); ax.plot(*meta, "r*", ms=14)
    ax.set_xlim(0, ARENA[0]); ax.set_ylim(0, ARENA[1]); ax.set_aspect("equal"); ax.set_title(titulo, fontsize=9)

# Primer escenario: trayectoria y señales de control
inicio_a, meta_a = (0.5, 0.5, 0.0), (5.0, 5.0)
res_a = ejecutar(inicio_a, meta_a)
print(f"llegó={res_a['llego']}  colisión={res_a['colision']}  t={res_a['tiempo']:.1f}s  "
      f"longitud={res_a['longitud']:.2f}m  holgura mín={res_a['holgura_min']:.2f}m")

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
dibujar(axs[0], res_a, "Trayectoria", meta_a, inicio_a)
t = np.arange(len(res_a["v"])) * DT
axs[1].plot(t, res_a["v"], label="v (m/s)"); axs[1].plot(t, res_a["w"], label="w (rad/s)")
axs[1].set_title("Salidas del controlador"); axs[1].set_xlabel("t (s)"); axs[1].legend()
axs[2].plot(t, res_a["d_frente"], label="frente"); axs[2].plot(t, res_a["d_izq"], label="izq")
axs[2].plot(t, res_a["d_der"], label="der"); axs[2].set_title("Lidar (sectores)"); axs[2].set_xlabel("t (s)"); axs[2].legend()
plt.tight_layout(); plt.show()

## 7. Defuzzificación → *Ítem 7*
(a) Efecto visual de cada método sobre una misma función agregada. (b) Los métodos aplicados al controlador, con las dos salidas. (c) Cómo cambia la trayectoria completa con centroide frente a *mom*.

In [ ]:
metodos = ["centroid", "bisector", "mom", "som", "lom"]

# (a) Visual
plt.figure(figsize=(8, 3.5))
plt.fill_between(x, y_or, alpha=.25, label="A OR B")
for m in metodos:
    plt.axvline(fuzz.defuzz(x, y_or, m), ls="--", label=m)
plt.legend(fontsize=8); plt.title("Defuzzificación sobre una función agregada"); plt.show()

# (b) Estado crítico: obstáculo al frente-derecha, meta a la izquierda
estado_critico = dict(d_frente=0.45, d_izq=0.90, d_der=0.30, err_rumbo=30.0)
resultados = {}
for m in metodos:
    v_out.defuzzify_method = m; w_out.defuzzify_method = m
    s = ctrl.ControlSystemSimulation(sistema)
    for k, val in estado_critico.items():
        s.input[k] = val
    s.compute()
    resultados[m] = (s.output["v"], s.output["w"])
v_out.defuzzify_method = "centroid"; w_out.defuzzify_method = "centroid"
print(pd.DataFrame(resultados, index=["v (m/s)", "w (rad/s)"]).T.round(3).to_string())

# (c) Trayectoria completa con dos métodos
fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))
comparacion = {}
for ax, m in zip(axs, ["centroid", "mom"]):
    r = ejecutar(inicio_a, meta_a, metodo=m)
    comparacion[m] = r
    dibujar(ax, r, f"{m}: llegó={r['llego']}, t={r['tiempo']:.0f}s, holgura={r['holgura_min']:.2f}m", meta_a, inicio_a)
plt.tight_layout(); plt.show()

## 8. Escenarios de prueba y superficies de control → *Ítem 8*

In [ ]:
ESCENARIOS = {
    "A: diagonal":            ((0.5, 0.5, 0.0),         (5.0, 5.0)),
    "B: esquina opuesta":     ((5.5, 0.5, np.pi),       (0.7, 5.3)),
    "C: desde pared":         ((0.5, 5.0, -np.pi / 2),  (5.2, 1.0)),
    "D: obstáculo al frente": ((1.0, 2.0, 0.0),         (5.0, 2.0)),
}
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
filas, res_esc = [], {}
for ax, (nombre, (ini, meta)) in zip(axs, ESCENARIOS.items()):
    r = ejecutar(ini, meta); res_esc[nombre] = r
    dibujar(ax, r, nombre, meta, ini)
    filas.append({"escenario": nombre, "llegó": r["llego"], "colisión": r["colision"],
                  "t (s)": round(r["tiempo"], 1), "longitud (m)": round(r["longitud"], 2),
                  "holgura mín (m)": round(r["holgura_min"], 2),
                  "v máx": round(r["v"].max(), 3), "|w| máx": round(np.abs(r["w"]).max(), 3)})
plt.tight_layout(); plt.show()
print(pd.DataFrame(filas).to_string(index=False))

# Superficies de control: camino libre a los lados (d_izq = d_der = 2 m)
gd = np.linspace(0.15, RANGO_MAX, 25); ge = np.linspace(-180, 180, 25)
simS = ctrl.ControlSystemSimulation(sistema)
Zv = np.zeros((len(ge), len(gd))); Zw = np.zeros_like(Zv)
for i, e in enumerate(ge):
    for j, d in enumerate(gd):
        simS.input["d_frente"] = d; simS.input["d_izq"] = 2.0; simS.input["d_der"] = 2.0
        simS.input["err_rumbo"] = e; simS.compute()
        Zv[i, j], Zw[i, j] = simS.output["v"], simS.output["w"]
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, Z, t in zip(axs, [Zv, Zw], ["v (m/s)", "w (rad/s)"]):
    c = ax.contourf(gd, ge, Z, 20, cmap="viridis"); plt.colorbar(c, ax=ax)
    ax.set_xlabel("d_frente (m)"); ax.set_ylabel("err_rumbo (°)"); ax.set_title(f"Superficie de control: {t}")
plt.tight_layout(); plt.show()

## 9. Autoverificación contra lo pedido por el profesor

In [ ]:
def chk(item, nombre, cond, nota=""):
    icono = "✅" if cond is True else ("⚠️" if cond is None else "❌")
    print(f"{icono} Ítem {item}: {nombre}" + (f"  → {nota}" if nota else ""))

def cobertura_ok():
    g = lambda var: np.linspace(var.universe.min(), var.universe.max(), 5)
    s = ctrl.ControlSystemSimulation(sistema)
    try:
        for a in g(d_frente):
            for b in g(d_izq):
                for c in g(d_der):
                    for e in g(err_rumbo):
                        s.input["d_frente"] = a; s.input["d_izq"] = b
                        s.input["d_der"] = c; s.input["err_rumbo"] = e
                        s.compute()
        return True
    except Exception as ex:
        print("   Error de cobertura:", ex); return False

ants, cons = [d_frente, d_izq, d_der, err_rumbo], [v_out, w_out]
llegaron = sum(r["llego"] for r in res_esc.values())
sin_choque = all(not r["colision"] for r in res_esc.values())
v_ok = all(r["v"].max() <= V_MAX + 1e-9 for r in res_esc.values())
w_ok = all(np.abs(r["w"]).max() <= W_MAX + 1e-9 for r in res_esc.values())

chk("2/3", "cada variable tiene ≥2 funciones de pertenencia", all(len(v.terms) >= 2 for v in ants + cons))
chk("3",   "catálogo de funciones y operadores AND/OR/NOT graficado", catalogo_ok)
chk("4",   "≥3 antecedentes y ≥1 consecuente", len(ants) >= 3 and len(cons) >= 1,
    f"{len(ants)} entradas, {len(cons)} salidas")
chk("5",   f"número de reglas ({len(reglas)}) entre 10 y 20", 10 <= len(reglas) <= 20)
chk("6",   "el sistema calcula salida en toda la rejilla de entradas (625 combinaciones)", cobertura_ok())
chk("6",   f"el robot llega a la meta en los escenarios ({llegaron}/{len(res_esc)})", llegaron == len(res_esc))
chk("6",   "ningún escenario termina en colisión", sin_choque)
chk("6",   "se respetan los límites v ≤ 0.22 m/s y |w| ≤ 1.0 rad/s", v_ok and w_ok)
chk("7",   "se compararon ≥3 métodos de defuzzificación", len(resultados) >= 3, ", ".join(resultados))
chk("8",   "se usaron datos reales del rosbag", True if USAR_CSV else None,
    "" if USAR_CSV else "datos SINTÉTICOS: pongan USAR_CSV = True")
print("\nManual: ítem 1 (hipótesis, sección 1) y que las funciones de pertenencia tengan sentido para SU robot.")
print("Recordatorio: entrega en grupo y presencial el 17 o 19 de noviembre, con rúbrica.")

## 10. Paso a ROS 2 (opcional, fuera del alcance de la entrega)
| En este notebook | En ROS 2 Humble |
|---|---|
| `lidar(x, y, yaw)` | suscriptor a `/scan` con los mismos 3 sectores |
| `err_rumbo` con pose simulada | pose de `/odom` y meta fija |
| `sim.output["v"], ["w"]` | `Twist` publicado en `/cmd_vel` |
| las 18 reglas | mismo código `skfuzzy` dentro del nodo |